# Preference Tuning, DPO, and Reward Modeling

Supervised fine-tuning teaches a model to imitate target answers. Preference tuning teaches a model to prefer better answers over worse ones. This is the layer behind methods such as DPO, ORPO, KTO, RLHF, PPO, and GRPO.

This notebook focuses on the concepts and mechanics, with DPO as the main hands-on method.

## What You Will Learn

By the end, you should understand:

- why preference tuning usually comes after SFT,
- what chosen/rejected data looks like,
- the intuition behind DPO,
- why a reference model is used,
- what reward modeling is,
- how RLHF/PPO differs from DPO,
- where ORPO, KTO, IPO, and GRPO fit,
- and how to evaluate preference-tuned models.

## 1. Why Preference Tuning Exists

SFT learns from one target answer. But real quality is often comparative: answer A is more correct, safer, clearer, or more useful than answer B. Preference tuning uses that comparison signal.

A common alignment pipeline is:

```text
base model -> supervised fine-tuning -> preference tuning -> evaluation -> serving
```

Preference tuning is not a replacement for good SFT. It usually refines a model that already knows how to answer.

## 2. Preference Method Map

| Method | Data | Main Idea | Notes |
|---|---|---|---|
| Reward modeling | prompt, chosen, rejected | Train a scorer for response quality | Often used before RLHF |
| RLHF with PPO | prompts plus reward model | Optimize model with reinforcement learning | Powerful but operationally complex |
| DPO | prompt, chosen, rejected | Directly optimize policy preferences | Simpler than PPO, widely used |
| IPO | prompt, chosen, rejected | Preference optimization with different regularization | DPO-family method |
| ORPO | prompt, chosen, rejected | Combines SFT and preference odds objective | Avoids separate reference model in some formulations |
| KTO | good/bad examples | Optimize from binary desirability labels | Useful when pairs are unavailable |
| GRPO | prompts and group rewards | PPO-style group relative optimization | Common in reasoning-model training discussions |

In [ ]:
import importlib.util
import math

import pandas as pd
import torch
from torch.nn import functional as F

def has_package(name):
    return importlib.util.find_spec(name) is not None

print(f"torch installed: {has_package('torch')}")
print(f"trl installed: {has_package('trl')}")
print(f"transformers installed: {has_package('transformers')}")

## 3. Chosen/Rejected Dataset

Preference data usually stores a prompt, a chosen response, and a rejected response. The chosen response should be better according to a clear rubric. It does not have to be perfect; it only has to be preferred over the rejected response.

In [ ]:
preference_records = [
    {
        "prompt": "Explain LoRA in two sentences.",
        "chosen": "LoRA freezes the base model and trains small low-rank adapter matrices. This gives strong task adaptation while updating far fewer parameters than full fine-tuning.",
        "rejected": "LoRA is just making the model smaller and faster.",
        "reason": "chosen is more accurate and specific",
    },
    {
        "prompt": "When should I use RAG instead of fine-tuning?",
        "chosen": "Use RAG when answers depend on external or frequently changing facts, and use fine-tuning when you need the model to learn a repeated behavior or style.",
        "rejected": "Always fine-tune because it stores all knowledge inside the model.",
        "reason": "chosen gives the correct trade-off",
    },
    {
        "prompt": "What does QLoRA save?",
        "chosen": "QLoRA saves GPU memory by loading the base model in 4-bit precision while training LoRA adapters.",
        "rejected": "QLoRA mainly saves disk space by deleting model layers.",
        "reason": "chosen correctly explains 4-bit loading and adapters",
    },
]
pd.DataFrame(preference_records)

## 4. Preference Data Quality

Preference data is only useful if the preference is meaningful. Bad preference pairs can train the model to prefer verbosity, flattery, unsafe behavior, or style over correctness.

In [ ]:
def preference_quality_report(records):
    rows = []
    for index, record in enumerate(records):
        chosen_words = len(record["chosen"].split())
        rejected_words = len(record["rejected"].split())
        rows.append({
            "index": index,
            "prompt_words": len(record["prompt"].split()),
            "chosen_words": chosen_words,
            "rejected_words": rejected_words,
            "chosen_longer": chosen_words > rejected_words,
            "has_reason": bool(record.get("reason")),
            "identical_pair": record["chosen"] == record["rejected"],
        })
    return pd.DataFrame(rows)

preference_quality_report(preference_records)

## 5. DPO Intuition

DPO, or Direct Preference Optimization, trains the model to assign higher probability to the chosen response than the rejected response, while staying close to a reference model.

The reference model is usually the SFT model before preference tuning. It acts as an anchor so the tuned model does not drift too far.

DPO compares log-probability gaps:

```text
policy_gap = logp_policy(chosen) - logp_policy(rejected)
reference_gap = logp_ref(chosen) - logp_ref(rejected)
loss = -logsigmoid(beta * (policy_gap - reference_gap))
```

If the policy improves the chosen-vs-rejected gap relative to the reference, the loss decreases.

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    policy_gap = policy_chosen_logps - policy_rejected_logps
    reference_gap = ref_chosen_logps - ref_rejected_logps
    logits = beta * (policy_gap - reference_gap)
    losses = -F.logsigmoid(logits)
    rewards_chosen = beta * (policy_chosen_logps - ref_chosen_logps)
    rewards_rejected = beta * (policy_rejected_logps - ref_rejected_logps)
    return losses, rewards_chosen, rewards_rejected

policy_chosen = torch.tensor([-12.0, -8.0, -10.5])
policy_rejected = torch.tensor([-14.0, -9.5, -13.0])
ref_chosen = torch.tensor([-12.5, -8.2, -10.8])
ref_rejected = torch.tensor([-13.0, -9.0, -12.0])

losses, chosen_rewards, rejected_rewards = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
pd.DataFrame({
    "loss": losses.tolist(),
    "chosen_reward": chosen_rewards.tolist(),
    "rejected_reward": rejected_rewards.tolist(),
    "reward_margin": (chosen_rewards - rejected_rewards).tolist(),
})

## 6. Beta Controls Preference Strength

The DPO `beta` parameter controls how strongly the model is pushed away from the reference. Higher beta can enforce preferences more aggressively, but may increase drift or instability.

In [ ]:
rows = []
for beta in [0.01, 0.05, 0.1, 0.5, 1.0]:
    losses, chosen_rewards, rejected_rewards = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=beta)
    rows.append({
        "beta": beta,
        "mean_loss": losses.mean().item(),
        "mean_reward_margin": (chosen_rewards - rejected_rewards).mean().item(),
    })
pd.DataFrame(rows)

## 7. DPO Trainer Skeleton

In practice, many teams use TRL's DPO trainer or similar tooling. This cell is a guarded skeleton: it shows the structure without requiring the package to be installed.

In [ ]:
try:
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import DPOConfig, DPOTrainer

    model_name = "sshleifer/tiny-gpt2"
    dpo_dataset = Dataset.from_list(preference_records)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)
    ref_model = AutoModelForCausalLM.from_pretrained(model_name)
    dpo_args = DPOConfig(
        output_dir="finetuning_outputs/dpo",
        per_device_train_batch_size=1,
        num_train_epochs=1,
        beta=0.1,
        report_to=[],
    )
    trainer = DPOTrainer(
        model=model,
        ref_model=ref_model,
        args=dpo_args,
        train_dataset=dpo_dataset,
        processing_class=tokenizer,
    )
    print("DPO trainer is ready. Uncomment trainer.train() when you want to run it.")
    # trainer.train()
except Exception as error:
    print("DPO trainer skeleton skipped. Install transformers datasets trl to run it.")
    print(f"Reason: {error}")

## 8. Reward Modeling

Reward modeling trains a separate model to score responses. Given a prompt and two responses, the reward model should assign a higher scalar score to the chosen response than the rejected response.

A simple pairwise reward loss is:

```text
loss = -logsigmoid(reward_chosen - reward_rejected)
```

This reward model can then guide RLHF with PPO or be used for reranking/evaluation.

In [ ]:
def reward_model_pairwise_loss(chosen_scores, rejected_scores):
    return -F.logsigmoid(chosen_scores - rejected_scores)

chosen_scores = torch.tensor([2.0, 1.2, 0.8])
rejected_scores = torch.tensor([0.5, 1.0, -0.2])
rm_losses = reward_model_pairwise_loss(chosen_scores, rejected_scores)
pd.DataFrame({
    "chosen_score": chosen_scores.tolist(),
    "rejected_score": rejected_scores.tolist(),
    "margin": (chosen_scores - rejected_scores).tolist(),
    "loss": rm_losses.tolist(),
})

## 9. RLHF, PPO, and GRPO at a High Level

| Stage | RLHF/PPO | GRPO-style training |
|---|---|---|
| Generate | sample model responses | sample groups of responses |
| Score | reward model scores each response | group-relative rewards or verifiable rewards |
| Optimize | PPO update with KL control | optimize relative to group rewards |
| Complexity | higher operational complexity | often used for reasoning/verifiable tasks |

For Week 5-6, know the map. You do not need to implement PPO or GRPO yet. DPO gives a much cleaner first preference-tuning path.

## 10. Evaluation for Preference Tuning

Preference tuning can make outputs sound better while becoming less factual. Evaluate with multiple signals:

- win rate against the SFT model,
- task correctness metrics,
- safety and refusal regression tests,
- verbosity and style checks,
- human side-by-side review,
- LLM-as-judge only with calibration and spot checks,
- latency and cost after deployment.

In [ ]:
eval_template = pd.DataFrame([
    {"prompt_id": "lora-basic", "sft_win": 0, "dpo_win": 1, "tie": 0, "notes": "DPO answer is more precise."},
    {"prompt_id": "rag-vs-ft", "sft_win": 0, "dpo_win": 0, "tie": 1, "notes": "Both answers are acceptable."},
])
summary = {
    "dpo_win_rate": eval_template["dpo_win"].mean(),
    "sft_win_rate": eval_template["sft_win"].mean(),
    "tie_rate": eval_template["tie"].mean(),
}
eval_template, summary

## 11. Common Preference Tuning Failure Modes

| Failure | What It Looks Like | Mitigation |
|---|---|---|
| Length bias | model prefers longer answers regardless of quality | balance pair lengths, evaluate conciseness |
| Style over truth | model sounds polished but is wrong | task metrics and factual checks |
| Reward hacking | model exploits reward model quirks | human review and adversarial evals |
| Over-refusal | model refuses safe prompts | refusal regression set |
| Under-refusal | model complies with unsafe prompts | safety eval set |
| Drift from SFT | model loses useful behavior | tune beta, use reference model, monitor regressions |
| Bad preference labels | preferences encode inconsistent taste | label guidelines and adjudication |

## 12. Decision Guide

| Situation | Method to Try |
|---|---|
| You have prompt-response examples only | SFT first |
| You have chosen/rejected pairs | DPO first |
| You have binary good/bad labels but not pairs | KTO-style methods |
| You need a reusable scorer | Reward model |
| You need online RL with rewards | PPO/RLHF or GRPO-style methods |
| You want simpler preference tuning after SFT | DPO or ORPO |

For most learning projects, do SFT, then DPO. Save RLHF/PPO/GRPO for when you have a strong reason and reliable reward signals.

## Summary

Preference tuning uses comparative data to push a model toward preferred responses. DPO is the best first method to learn because it directly optimizes chosen-over-rejected pairs without building a full RLHF stack. Reward modeling and PPO/GRPO are important concepts, but they are more complex and should come after strong SFT, good evals, and clean preference data.